# Lab 4
- **Student Name**: Daniel Alejandro Castillo Martin
- **Student ID**: N10019284
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
---

In this lab, we investigate memory and gradient propagation in recurrent neural networks using a delayed-recall task. Controlled experiments compare plain RNNs, LSTMs, and GRUs, followed by measurements of gradients through time, gate initialization, gradient clipping, and computation time.

## 1. Imports and Setup

In [14]:
# Import required libraries and modules
import time
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

# Import starter script
scripts_dir = Path('scripts').resolve()
if not (scripts_dir / 'lab04_starter.py').is_file():
    raise FileNotFoundError('Run notebook from the LAB-04 repository root.')
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from lab04_starter import make_recall_batch, RecallNet, train_recall

# See current PyTorch version
print('PyTorch Version:', torch.__version__)

PyTorch Version: 2.14.0+cpu


In [ ]:
# Output folder paths
RESULTS_DIR = Path('output/results')
FIGURES_DIR = Path('output/figures')

FIGURES_DIR.mkdir(parents = True, exist_ok = True)
RESULTS_DIR.mkdir(parents = True, exist_ok = True)

# Seeds for training comparisons
SEEDS = [0, 1, 2]

In [9]:
# Show device type in which the notebook is being executed
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


The professor's starter code is preserved and imported from the `scripts/` folder. The experiments use its CPU training implementation and default training settings. Each training run sets its own seed through `train_recall`.

First and before any experiment, the starter code is executed to validate that the plain RNN solves the 10-step delayed-recall task.

In [ ]:
# Run the starter code as a separate Python process
starter_check = subprocess.run([sys.executable, str(scripts_dir / 'lab04_starter.py')], capture_output = True, text = True)
print(starter_check.stdout)

if starter_check.stderr:
    print(starter_check.stderr)

starter_check.check_returncode()

Plain recurrent layer, 10 steps, 600 updates: validation accuracy 1.000 (chance is 0.25)
Starter OK.



## 2. Recurrent Benchmark

- Five recurrent configurations are compared at sequence lengths **10**, **30**, and **60** using seeds **0**, **1**, and **2**.
- All runs preserve the starter's defaults: hidden size **32**, **600** updates, batch size **64**, Adam optimizer, learning rate **0.003**, and gradient clipping at **1.0**.
- For each sequence length, the validation set is fixed. For each seed and length, all configurations receive the same training batches.
- Validation accuracy is reported for every seed and as a mean.
- Chance accuracy is **0.25**.

In [23]:
# Define the five required configurations
CONFIGS = [
    ('RNN', 'rnn', None),
    ('LSTM default', 'lstm', None),
    ('LSTM bias 3', 'lstm', 3.0),
    ('GRU default', 'gru', None),
    ('GRU bias 3', 'gru', 3.0)
]

SEQ_LENGTHS = [10, 30, 60]

pd.DataFrame(CONFIGS, columns=['Configuration', 'Cell type', 'Gate bias'],).fillna('Default')

,Configuration,Cell type,Gate bias
0,RNN,rnn,Default
1,LSTM default,lstm,Default
2,LSTM bias 3,lstm,3.0
3,GRU default,gru,Default
4,GRU bias 3,gru,3.0


In [ ]:
# Verify selected gate slices before training
for kind in ['lstm', 'gru']:
    torch.manual_seed(0)

    biased_model = RecallNet(kind, gate_bias = 3.0)

    hidden = biased_model.rnn.hidden_size
    gate_slice = slice(hidden, 2 * hidden)

    bias_ih = biased_model.rnn.bias_ih_l0[gate_slice].detach()
    bias_hh = biased_model.rnn.bias_hh_l0[gate_slice].detach()
    effective_bias = bias_ih + bias_hh

    gate_name = 'forget' if kind == 'lstm' else 'update'

    print(f'\n{kind.upper()} {gate_name} gate: [{hidden}:{2 * hidden}]')
    print('bias_ih slice:', bias_ih)
    print('bias_hh slice:', bias_hh)
    print('Effective bias:', effective_bias)

    assert torch.allclose(bias_ih, torch.full_like(bias_ih, 1.5))
    assert torch.allclose(bias_hh, torch.full_like(bias_hh, 1.5))
    assert torch.allclose(effective_bias,torch.full_like(effective_bias, 3.0))

print('\nGate-bias checks passed.')


LSTM forget gate: [32:64]
bias_ih slice: tensor([1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000])
bias_hh slice: tensor([1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000])
Effective bias: tensor([3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3.,
        3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3., 3.])

GRU update gate: [32:64]
bias_ih slice: tensor([1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
   

In [24]:
# Train every required configuration, length, and seed
benchmark_rows = []
benchmark_history_rows = []

total_runs = len(CONFIGS) * len(SEQ_LENGTHS) * len(SEEDS)
run_number = 0

for name, kind, gate_bias in CONFIGS:
    for seq_len in SEQ_LENGTHS:
        for seed in SEEDS:
            run_number += 1

            print(f'[{run_number}/{total_runs}] ' f'{name}, length={seq_len}, seed={seed}', flush = True)

            accuracy, history = train_recall(kind, seq_len = seq_len, seed = seed, gate_bias = gate_bias,)
            benchmark_rows.append({
                'configuration': name,
                'seq_len': seq_len,
                'seed': seed,
                'accuracy': accuracy
                })

            for update, val_accuracy in history:
                benchmark_history_rows.append({
                    'configuration': name,
                    'seq_len': seq_len,
                    'seed': seed,
                    'update': update,
                    'accuracy': val_accuracy
                    })

            # Save completed results after each run
            pd.DataFrame(benchmark_rows).to_csv(RESULTS_DIR / 'benchmark_runs.csv', index = False)
            pd.DataFrame(benchmark_history_rows).to_csv(RESULTS_DIR / 'benchmark_history.csv', index = False)
            print(f'Final validation accuracy: {accuracy:.3f}')

benchmark_df = pd.DataFrame(benchmark_rows)

print('\nBenchmark complete.')

[1/45] RNN, length=10, seed=0
Final validation accuracy: 1.000
[2/45] RNN, length=10, seed=1
Final validation accuracy: 1.000
[3/45] RNN, length=10, seed=2
Final validation accuracy: 0.495
[4/45] RNN, length=30, seed=0
Final validation accuracy: 0.248
[5/45] RNN, length=30, seed=1
Final validation accuracy: 0.251
[6/45] RNN, length=30, seed=2
Final validation accuracy: 0.264
[7/45] RNN, length=60, seed=0
Final validation accuracy: 0.250
[8/45] RNN, length=60, seed=1
Final validation accuracy: 0.240
[9/45] RNN, length=60, seed=2
Final validation accuracy: 0.233
[10/45] LSTM default, length=10, seed=0
Final validation accuracy: 1.000
[11/45] LSTM default, length=10, seed=1
Final validation accuracy: 1.000
[12/45] LSTM default, length=10, seed=2
Final validation accuracy: 1.000
[13/45] LSTM default, length=30, seed=0
Final validation accuracy: 0.248
[14/45] LSTM default, length=30, seed=1
Final validation accuracy: 0.251
[15/45] LSTM default, length=30, seed=2
Final validation accuracy: 0

In [26]:
# Show every seed and the mean in one table
benchmark_table = benchmark_df.pivot(index = ['configuration', 'seq_len'], columns = 'seed', values = 'accuracy',).reindex(columns = SEEDS)

# Preserve the intended configuration and length order
row_order = pd.MultiIndex.from_product([[name for name, _, _ in CONFIGS], SEQ_LENGTHS], names = ['configuration', 'seq_len'],)

benchmark_table = benchmark_table.reindex(row_order)
benchmark_table.columns = [f'seed_{seed}' for seed in SEEDS]
benchmark_table['mean'] = benchmark_table.mean(axis=1)

benchmark_table.to_csv(RESULTS_DIR / 'benchmark_summary.csv')

benchmark_table.style.format('{:.3f}')

- At sequence length 10, both LSTM and GRU configurations achieved **1.000** validation accuracy for every seed. The plain RNN achieved **1.000** for seeds **0** and **1**, but only **0.495** for seed **2**.
- At lengths **30** and **60**, the default RNN, LSTM, and GRU configurations remained close to chance accuracy of **0.25**. Both gated configurations with bias **3** achieved **1.000** across all tested lengths and seeds.
- Under these training conditions, gated architectures with default initialization did not reliably learn the longer recall tasks. The gradient experiment will investigate the propagation mechanism that may explain the initialization-dependent performance.